# M3_03 — shared configuration, deterministic prompts and token budgeting

This notebook explains and verifies M3-03 without running model weights. CPU tests use explicit test counters. An optional offline check uses the real Qwen3-8B tokenizer and original chat template; **no GPU is required**.

Configuration source: `config/runtime_defaults.json`. Both `config/dev.json` and `config/run_config.json` inherit it via relative `extends`. Future jobs should load these files with `load_config`, not duplicate constants. The resolved config hash records overrides.

**Implemented:** shared serving/context limits; local cached tokenizer; per-stage input budgets; deterministic README/schema/few-shot reduction; no-call overflow; parse/solve/repair integration; factual budget metadata in ModelCall; server-argument helper for M4.
**Outside scope:** model quality, learned retrieval, few-shot-bank construction (M3-10), server lifecycle, held-out PA and GPU performance.

PASS = observed criterion passed. FAIL = an observed check failed. PARTIAL = some coverage skipped/missing. NOT VERIFIED = optional real-tokenizer check was not run. These labels never imply target GPU/Docker acceptance.

**Observed local validation (2026-11-11):** 19 M3-03 cases passed; full CPU suite 233 passed / 2 skipped (training data unavailable); Ruff passed; all 39 fixture parse/solve/repair plans passed with the pinned real Qwen tokenizer. Test/report/export cells were executed using the existing checkout/dependencies and plain-text display instead of Colab clone/install. JSON/code syntax and CSV/JSON/ZIP exports were checked. This is recorded evidence, not pre-filled outputs for your future run. Details and shared-interface notice: SPEC section 18.6.


## 1. Changes, reasons and boundaries

| Area | Implementation | Reason |
|---|---|---|
| Shared profile | runtime_defaults.json + config extends | Change context/output/budgets once; jobs resolve the same values |
| Derived limits | RunConfig.input_budget(stage) | Avoid duplicated hard-coded 7104/6080 limits |
| Local tokenizer | llm/tokenizer.py | Load once, local_files_only=True, count full chat template and thinking setting |
| Prompt preparation | prompts/budget.py | Preserve mandatory context; apply deterministic reductions |
| Execution integration | parse/parser.py, solver/solve.py | Check before every call, including entire repair history |
| Overflow evidence | E_CONTEXT_OVERFLOW; finish_reason=not_sent | Abstain without pretending the model was called |
| Additive contracts 1.3.0 | PromptPlan; optional ModelCall.prompt_budget | Record counts, reductions, prefix scope/hash and whether client.complete was invoked |
| M4 adapter | server_arguments(cfg) | Build launch arguments from the same limits without starting a server |

Normal prompts retain the existing template wording/IDs. Runtime context reductions are tracked by policy version. Unreduced/business-capped prefixes are byte-identical for a business/template. Question-specific README/schema reductions explicitly use prefix_scope=reduced; they do not claim a universal business prefix.

Rulebook, original question, QuestionFrame, windows, and previous repair messages are never cut. Full BusinessCatalog remains intact for execution/business logs. Relevant-subset fallback preserves full columns for up to four selected tables and names of others; it is not proof the chosen schema contains every necessary join.

CPU scripted mode uses a clearly labeled UTF-8-byte stand-in, not fake model-token measurements. HTTP production requires a local tokenizer and validates readiness before result files are opened.


## 2. Checkout and environment

Fill the URL for a clean commit containing M3-03. REPO_REF accepts a branch, tag or commit SHA; prefer a SHA for recorded evidence. The notebook fetches that reference and checks out FETCH_HEAD. No source files are edited. Logs and exports go outside the checkout under /content/m3_03_checks. CPU tests need no weights/network after dependency setup. Real token counting additionally needs the tokenizer extra (Transformers + Jinja2); that extra does not load a model.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, subprocess, sys, xml.etree.ElementTree as ET

REPO_URL = ""  # REQUIRED
REPO_REF = ""  # Prefer an immutable commit containing M3-03.
REPO_DIR = Path("/content/DataMate")
RESULT_DIR = Path("/content/m3_03_checks") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RESULT_DIR.mkdir(parents=True, exist_ok=False)
RUN_REAL_TOKENIZER_CHECK = False
DOWNLOAD_TOKENIZER = False  # Optional setup download, tokenizer/config only.
TOKENIZER_PATH = ""
TOKENIZER_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"

def command(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, text=True, **kwargs)
def save_json(name, value):
    path = RESULT_DIR / name
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    return path

if not REPO_URL.strip():
    raise ValueError("Fill REPO_URL before running.")
if not REPO_DIR.exists():
    command(["git", "clone", "--", REPO_URL, REPO_DIR])
if subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Use a clean checkout for reproducible evidence.")
if REPO_REF:
    command(["git", "-C", REPO_DIR, "fetch", "origin", REPO_REF])
    command(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
command([sys.executable, "-m", "pip", "install", "-r", REPO_DIR / "requirements-cpu.lock"])
command([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "--no-deps"])
sys.path.insert(0, str(REPO_DIR / "src"))
TEST_ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}
from IPython.display import display, Markdown
from datamate.config import load_config
from datamate.contracts import CONTRACTS_VERSION
from datamate.llm import get_token_counter, server_arguments
from datamate.prompts import BUDGET_POLICY_VERSION, prepare_parse, prepare_solve
PROFILE = REPO_DIR / "config/runtime_defaults.json"
cfg = load_config(PROFILE)
print("Commit:", REPO_COMMIT, "| contracts:", CONTRACTS_VERSION, "| profile:", PROFILE)
save_json("effective_config.json", cfg.model_dump(mode="json"))


## 3. Shared limits and future reconfiguration

The shipped profile uses context 8192, outputs 1024/2048/2048, safety 64, README 512 and schema 3000. Input limits are **derived** as 7104/6080/6080. Context is prompt+output; max_num_batched_tokens is a scheduler budget, not a per-request context.

This cell shows how changing only serving.max_model_len to 16384 recomputes limits and changes the config hash. It does not certify that 16K fits the GPU. Overrides below are in memory, recorded in exported evidence, and do not edit the shared file.


In [ ]:
def show_rows(rows):
    keys = list(rows[0]) if rows else []
    if not keys:
        display(Markdown("_No rows._")); return
    def clean(v): return str(v).replace("|", "/").replace("\n", " ")
    text = "| " + " | ".join(keys) + " |\n| " + " | ".join("---" for _ in keys) + " |\n"
    text += "\n".join("| " + " | ".join(clean(row.get(k, "")) for k in keys) + " |" for row in rows)
    display(Markdown(text))
larger = load_config(PROFILE, ["serving.max_model_len=16384"])
limit_rows = [
    dict(stage=stage, context=cfg.serving.max_model_len, output=cfg.model.max_tokens[stage],
         safety=cfg.prompt_budget.safety_margin_tokens, input_limit=cfg.input_budget(stage),
         input_limit_at_16K=larger.input_budget(stage))
    for stage in ("parse", "solve", "repair")
]
show_rows(limit_rows)
assert [r["input_limit"] for r in limit_rows] == [7104, 6080, 6080]
assert larger.input_budget("solve") == 14272 and larger.sha256() != cfg.sha256()
save_json("limits.json", limit_rows)
launch_cfg = load_config(PROFILE, ["model.path=/local/pinned/model",
                                  "model.base_url=http://127.0.0.1:8000/v1"])
print("Illustrative M4 arguments (not executed):", server_arguments(launch_cfg, total_vram_mib=80*1024))
save_json("server_args_example.json", server_arguments(launch_cfg, total_vram_mib=80*1024))


## 4. CPU test runner

Each criterion below executes repository tests and saves stdout plus JUnit XML. PASS requires collected tests, exit code zero and no failures/skips. Skipped coverage is PARTIAL; collection/runtime/assertion errors are FAIL. Test counts are not real-model token counts.


In [ ]:
CHECKS = []
def check(key, title, names=None, paths=None):
    xml_path = RESULT_DIR / f"{key}.xml"
    args = [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
            "--junitxml", str(xml_path), *(paths or ["tests/unit/test_prompt_budget.py"])]
    if names: args += ["-k", " or ".join(names)]
    result = subprocess.run(args, cwd=REPO_DIR, env=TEST_ENV, capture_output=True, text=True, timeout=300)
    (RESULT_DIR / f"{key}.log").write_text(result.stdout + result.stderr, encoding="utf-8")
    cases = ET.parse(xml_path).getroot().findall(".//testcase") if xml_path.exists() else []
    failed = sum(c.find("failure") is not None or c.find("error") is not None for c in cases)
    skipped = sum(c.find("skipped") is not None for c in cases)
    status = "FAIL" if result.returncode or failed or not cases else "PARTIAL" if skipped else "PASS"
    row = dict(criterion=title, status=status, passed=len(cases)-failed-skipped,
               total=len(cases), failed=failed, skipped=skipped)
    CHECKS.append(row); show_rows([row]); print(result.stdout[-2500:])
    if result.returncode: print(result.stderr[-2500:])
    return row


## 5. Criterion A — shared configuration and invalid settings

Tests cover relative inheritance/deep merge, cycle rejection, derived limits after a context change, invalid output/safety budgets, and M4 launch arguments reading those same limits. PASS proves the config contract works; it does not prove an external server was launched with it. No hard-coded 6080 is used by the production budgeter.


In [ ]:
check("configuration", "Shared config + limits + M4 argument adapter", [
    "test_shared_profile_and_derived_limits",
    "test_config_inheritance_merges_sections_and_rejects_cycles",
    "test_invalid_context_config_fails_early",
    "test_server_arguments_use_same_context_config",
])


## 6. Criterion B — deterministic rendering and reductions

Render identical inputs twice and compare bytes. Different questions retain an identical normal business prefix. Section truncation preserves Unicode and inserts a marker where space permits. Few-shots are dropped last-first before dynamic README reduction. Large schema cards use deterministic lexical ranking and stable name ties; the full catalog is never mutated.

PASS means these mechanics matched independent fixture expectations. It does not prove subset selection improves PA. Prefix scope=reduced is an explicit exception to the normal shared prefix, not a false prefix-caching PASS.


In [ ]:
check("rendering", "Determinism + reduction order + immutable catalog", [
    "test_normal_render_and_business_prefix_are_stable",
    "test_solve_keeps_rulebook_frame_windows_and_catalog_unchanged",
    "test_readme_section_cap_has_marker_and_preserves_unicode",
    "test_drop_fewshots_last_first_before_readme",
    "test_readme_reduction_follows_fewshots",
    "test_relevant_schema_uses_metadata_and_stable_ties",
])


## 7. Criterion C — boundaries, mandatory context and repair

Exact-fit input is accepted. Mandatory content that cannot fit yields E_CONTEXT_OVERFLOW before invoking the model. The record explicitly has finish_reason=not_sent, no generated reasoning and client_called=false. Repair recounts context + previous generated output + error before the next call; it never silently drops those messages.

PASS here means no oversize request escaped in the tested cases. An overflow is an intentional abstain path, not a fabricated model answer.


In [ ]:
check("overflow_repair", "Exact limits + no-call overflow + repair history", [
    "test_exact_boundary_and_one_unit_overflow",
    "test_overflow_preserves_mandatory_context_and_does_not_call_client",
    "test_repair_history_is_recounted_before_next_call",
])


## 8. Criterion D — local tokenizer setup and development isolation

CPU unit tests use a controlled tokenizer double to verify local_files_only=True, trust_remote_code=False, chat-template/generation/thinking arguments and caching. Missing production tokenizer assets fail startup before result files are created. Scripted runs use a visibly labeled development counter; HTTP production has no automatic byte-count fallback.

PASS does not by itself verify Qwen's real tokenizer; use the optional section below for that evidence.


In [ ]:
check("tokenizer_setup", "Offline tokenizer wiring + cache + startup guards", [
    "test_local_counter_is_offline_cached_and_uses_chat_template",
    "test_missing_tokenizer_fails_before_outputs",
    "test_development_counter_is_explicit_and_not_used_for_http",
])


## 9. Regression and lint

The full CPU suite checks that M3-02 HTTP, contracts, temporal logic, formatting and scripted integration still work. Optional training-reference skips are reported as PARTIAL. Lint checks src/tests without installing GPU libraries.


In [ ]:
check("regression", "Full CPU regression", paths=["tests"])
lint = subprocess.run([sys.executable, "-m", "ruff", "check", "src", "tests", "--no-cache"],
                      cwd=REPO_DIR, env=TEST_ENV, capture_output=True, text=True)
(RESULT_DIR / "lint.log").write_text(lint.stdout + lint.stderr, encoding="utf-8")
CHECKS.append(dict(criterion="Ruff", status="PASS" if lint.returncode==0 else "FAIL",
                  passed=int(lint.returncode==0), total=1, failed=int(lint.returncode!=0), skipped=0))
show_rows([CHECKS[-1]]); print(lint.stdout, lint.stderr)


## 10. Optional real tokenizer assets — no GPU, no model weights

Set RUN_REAL_TOKENIZER_CHECK=True. If TOKENIZER_PATH already points to a complete pinned local snapshot, leave DOWNLOAD_TOKENIZER=False. Otherwise set DOWNLOAD_TOKENIZER=True to download only tokenizer/config files from the fixed Qwen3-8B revision.

Internet is used only during this preparation cell. Actual counting uses local_files_only=True. The optional extra includes Jinja2 because chat-template rendering needs it even without PyTorch/vLLM.

Source: [pinned Qwen3-8B snapshot](https://huggingface.co/Qwen/Qwen3-8B/tree/b968826d9c46dd6066d109eabc6255188de91218).


In [ ]:
TOKENIZER_SETUP_ERROR = None
REAL_ROWS = []
if RUN_REAL_TOKENIZER_CHECK:
    try:
        command([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR) + "[tokenizer]"])
        if DOWNLOAD_TOKENIZER:
            from huggingface_hub import snapshot_download
            TOKENIZER_PATH = snapshot_download("Qwen/Qwen3-8B", revision=TOKENIZER_REVISION,
                cache_dir=str(RESULT_DIR.parent / "tokenizer_cache"),
                allow_patterns=["tokenizer*", "config.json", "vocab.json", "merges.txt",
                                "special_tokens_map.json", "added_tokens.json", "*.jinja"])
        if not TOKENIZER_PATH:
            raise ValueError("Set TOKENIZER_PATH or enable DOWNLOAD_TOKENIZER.")
        real_cfg = load_config(PROFILE, [f"model.path={TOKENIZER_PATH}"])
        real_counter = get_token_counter(real_cfg)
        save_json("real_tokenizer_fingerprint.json", real_counter.fingerprint)
        print(real_counter.fingerprint)
    except Exception as exc:
        TOKENIZER_SETUP_ERROR = f"{type(exc).__name__}: {exc}"
        print(TOKENIZER_SETUP_ERROR)
else:
    print("Real tokenizer: NOT VERIFIED (disabled).")


## 11. Real tokenizer criterion — exact chat-template counts and limits

For each fixture question, prepare parse, solve and repair contexts. Compare the production counter with an independently loaded AutoTokenizer applying the same original chat template and thinking flag. Every accepted plan must fit input + output + safety <= server context and preserve the entire rulebook.

PASS means all measured counts/limits agree and contexts fit on this synthetic workload. An overflow is recorded as OVERFLOW (the input was correctly refused); it does not prove a broken counter. NOT VERIFIED means disabled/setup unavailable. This test never calls an LLM or claims GPU memory, latency or answer accuracy.


In [ ]:
if RUN_REAL_TOKENIZER_CHECK and TOKENIZER_SETUP_ERROR is None:
    from transformers import AutoTokenizer
    from datamate.catalog import setup_business
    from datamate.contracts import Question, QuestionFrame
    from datamate.prompts import render_repair
    from datamate.rules import load_split
    from datamate.temporal import resolve_windows
    from datetime import date
    oracle = AutoTokenizer.from_pretrained(TOKENIZER_PATH, local_files_only=True, trust_remote_code=False)
    split_path = REPO_DIR / "tests/fixtures/split_mini"
    split = load_split(split_path)
    scripts = [json.loads(s) for s in
        (REPO_DIR / "tests/fixtures/scripted/split_mini.jsonl").read_text(encoding="utf-8").splitlines()
        if s.strip()]
    frames = {e["question"]: QuestionFrame.model_validate_json(e["content"])
              for e in scripts if e["stage"]=="parse"}
    catalogs = {}
    for i,line in enumerate((split_path/"questions.jsonl").read_text(encoding="utf-8").splitlines(),1):
        raw = json.loads(line)
        q = Question(**{**raw, "as_of_date":date.fromisoformat(raw["as_of_date"]), "line_no":i})
        if q.business_id not in catalogs:
            catalogs[q.business_id] = setup_business(split_path/"data"/q.business_id,
                                                     q.business_id, RESULT_DIR/"work")
        cat = catalogs[q.business_id]
        frame = frames[q.question]
        windows,error = resolve_windows(frame.time,q.as_of_date,split.temporal,cat.data_range)
        plans = [("parse",prepare_parse(q,split,cat,real_cfg))]
        if error is None:
            plans += [("solve",prepare_solve(q,frame,windows,split,cat,real_cfg))]
            history = render_repair([], '{"sql":"SELECT missing_column FROM ban_hang"}',
                                    "E_SQL_EXEC", "Referenced column missing_column was not found.")
            plans += [("repair",prepare_solve(q,frame,windows,split,cat,real_cfg,
                                               stage="repair",history=history))]
        for stage,plan in plans:
            independent = len(oracle.apply_chat_template(plan.messages,tokenize=True,
                add_generation_prompt=True,enable_thinking=real_cfg.model.enable_thinking.get(stage,False)))
            counts_match = independent == plan.diagnostics["input_tokens"]
            fits = independent + real_cfg.model.max_tokens[stage] + real_cfg.prompt_budget.safety_margin_tokens <= real_cfg.serving.max_model_len
            rulebook_kept = split.rulebook_text in plan.messages[0]["content"]
            status = ("FAIL" if not counts_match or not rulebook_kept or (plan.error_code is None and not fits)
                      else "OVERFLOW" if plan.error_code else "PASS")
            REAL_ROWS.append(dict(id=q.id,stage=stage,status=status,tokens=independent,
                input_limit=real_cfg.input_budget(stage),counts_match=counts_match,
                fits=fits,rulebook_kept=rulebook_kept,prefix_scope=plan.diagnostics["prefix_scope"],
                actions=plan.diagnostics["actions"]))
    assert get_token_counter(real_cfg) is real_counter
    show_rows(REAL_ROWS)
    save_json("real_tokenizer_checks.json",REAL_ROWS)
else:
    REAL_ROWS = [dict(status="NOT VERIFIED",reason=TOKENIZER_SETUP_ERROR or "disabled")]
    show_rows(REAL_ROWS)


## 12. Summary and export

The summary separates fake-counter CPU acceptance from actual Qwen-tokenizer evidence. No automatic task-Done, GPU or model-quality claim is made. Keep the shared profile, resolved config hash, policy version, tokenizer fingerprint and source commit with each run.

The production pipeline records per-call prompt_budget diagnostics: input/output limits, counter identity, reductions, prefix hash/scope and client_called. E_CONTEXT_OVERFLOW carries no generated response and an empty request hash because no request was sent.

Before changing model/context/output/concurrency later, update the shared profile (or recorded dev overrides), derive M4 launch arguments from it, and re-run these checks. This helper cannot enforce an external server launched with unrelated manual flags.


In [ ]:
show_rows(CHECKS)
save_json("cpu_checks.json",CHECKS)
save_json("summary.json",dict(repo_commit=REPO_COMMIT,contracts=CONTRACTS_VERSION,
    config_sha256=cfg.sha256(),policy=BUDGET_POLICY_VERSION,cpu=CHECKS,real=REAL_ROWS,
    tokenizer_setup_error=TOKENIZER_SETUP_ERROR))
with (RESULT_DIR/"criteria.csv").open("w",newline="",encoding="utf-8-sig") as fh:
    writer=csv.DictWriter(fh,fieldnames=list(CHECKS[0]));writer.writeheader();writer.writerows(CHECKS)
freeze=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)
(RESULT_DIR/"requirements.txt").write_text(freeze,encoding="utf-8")
(RESULT_DIR/"CONCLUSION.md").write_text(
    "# M3-03 review\n\n"
    "- Shared profile/overrides reviewed with M4: [fill]\n"
    "- CPU failed/skipped checks: [fill]\n"
    "- Real tokenizer evidence: [fill]\n"
    "- Normal vs reduced prefix behavior reviewed with M1/Lead: [fill]\n"
    "- Overflow/reduction impact on held-out accuracy: NOT VERIFIED\n"
    "- Actual GPU/server configuration match: NOT VERIFIED\n"
    "- Reviewer decision: [fill]\n",encoding="utf-8")
print("Evidence:",RESULT_DIR)


## 13. Optional download

Archive includes logs, criteria and tokenizer checks, excluding temporary databases and downloaded tokenizer cache. No weights were downloaded. Colab files are ephemeral; preserve the evidence before ending the session.


In [ ]:
import zipfile
archive=RESULT_DIR.parent/f"{RESULT_DIR.name}.zip"
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(RESULT_DIR.rglob("*")):
        if path.is_file() and "work" not in path.relative_to(RESULT_DIR).parts:
            output.write(path,str(path.relative_to(RESULT_DIR)))
print(archive)
try:
    from google.colab import files
except ImportError:
    print("Use the local file browser to download.")
else:
    files.download(str(archive))
